# Lab 1: Neural Network Foundations

Capacity can help a network fit its training data, but useful capacity must
also be optimized and must generalize. In this lab you will work with a
**complete, runnable MLP and training loop** on CIFAR-10. Your job is to
change one experimental factor at a time, inspect learning curves, and make
defensible model-development decisions.

This online adaptation is based on MIT 6.7960 Fall 2024 HW1. The original
notebook's two 8,192-unit hidden layers create a 92-million-parameter model
and its missing-code exercises are intentionally not reproduced here. The
default below is much smaller and all implementations are prefilled.

**Learning objectives**

- Relate hidden width and depth to parameter count and empirical capacity.
- Distinguish underfitting from overfitting using training and validation curves.
- Test dropout, L2 weight decay, and train-only augmentation in controlled runs.
- Keep the official CIFAR-10 test split out of tuning decisions.


## How this notebook is assessed

Run the experiments and use the plots as evidence for your own reflection.
The MIT Learn submission grades one compact deterministic report and two
practitioner decisions. **No freely measured accuracy or loss is graded**:
those values can vary with hardware and library versions.

Quick mode is on by default and uses a deterministic 5,000-image training
subset plus a disjoint 1,000-image validation subset. Full mode uses the
entire 50,000-image development set as a 40,000/10,000 split. In both modes,
the official 10,000-image CIFAR-10 test set remains untouched.


In [ ]:
from __future__ import annotations

import math
import random
import time
from dataclasses import asdict, dataclass, replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 7960
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def reset_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # These settings improve repeatability. Small numerical differences can
    # still occur across devices and library versions, so metrics are ungraded.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


reset_seed()
print(f"PyTorch {torch.__version__}; device = {DEVICE}")


In [ ]:
# Set QUICK_MODE = False for the larger development split and longer runs.
QUICK_MODE = True
DATA_ROOT = "./data"

TRAIN_PER_CLASS = 500 if QUICK_MODE else 4_000
VALIDATION_PER_CLASS = 100 if QUICK_MODE else 1_000
BATCH_SIZE = 256
BASELINE_EPOCHS = 5 if QUICK_MODE else 12
EXPERIMENT_EPOCHS = 4 if QUICK_MODE else 8
NUM_WORKERS = 0 if QUICK_MODE else 2

print(
    f"quick_mode={QUICK_MODE}, train={10 * TRAIN_PER_CLASS:,}, "
    f"validation={10 * VALIDATION_PER_CLASS:,}, batch={BATCH_SIZE}"
)


## 1. Prepare controlled train and validation splits

CIFAR-10 images are RGB tensors with shape 3×32×32. The source notebook
accidentally mentions 28×28 once, but its code—and this lab—correctly use
32×32 images, so a flattened input has 3,072 values.

The split below is stratified and deterministic. Plain and augmented dataset
views share exactly the same indices. Augmentation is applied **only** to the
training view; validation data receive deterministic normalization only.


In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

plain_transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)]
)
augmented_transform = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=3, padding_mode="reflect"),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ]
)

# This transform-free instance is used only to read class targets.
split_source = datasets.CIFAR10(DATA_ROOT, train=True, download=True)


def stratified_split(
    targets: list[int], train_per_class: int, validation_per_class: int, seed: int
) -> tuple[list[int], list[int]]:
    target_tensor = torch.as_tensor(targets)
    generator = torch.Generator().manual_seed(seed)
    train_indices: list[int] = []
    validation_indices: list[int] = []
    for class_id in range(10):
        class_indices = torch.where(target_tensor == class_id)[0]
        order = torch.randperm(len(class_indices), generator=generator)
        class_indices = class_indices[order]
        needed = train_per_class + validation_per_class
        if needed > len(class_indices):
            raise ValueError(f"Requested {needed} examples from class {class_id}")
        train_indices.extend(class_indices[:train_per_class].tolist())
        validation_indices.extend(
            class_indices[train_per_class:needed].tolist()
        )
    return sorted(train_indices), sorted(validation_indices)


train_indices, validation_indices = stratified_split(
    split_source.targets, TRAIN_PER_CLASS, VALIDATION_PER_CLASS, SEED
)
assert set(train_indices).isdisjoint(validation_indices)

plain_source = datasets.CIFAR10(
    DATA_ROOT, train=True, transform=plain_transform, download=False
)
augmented_source = datasets.CIFAR10(
    DATA_ROOT, train=True, transform=augmented_transform, download=False
)

plain_train_set = Subset(plain_source, train_indices)
augmented_train_set = Subset(augmented_source, train_indices)
validation_set = Subset(plain_source, validation_indices)


def seeded_loader(
    dataset,
    *,
    batch_size: int,
    shuffle: bool,
    seed: int = SEED,
) -> DataLoader:
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        generator=generator,
    )


def make_m1_loaders(
    *, use_augmentation: bool, batch_size: int, seed: int = SEED
) -> tuple[DataLoader, DataLoader, DataLoader]:
    fit_set = augmented_train_set if use_augmentation else plain_train_set
    fit_loader = seeded_loader(
        fit_set, batch_size=batch_size, shuffle=True, seed=seed
    )
    train_eval_loader = seeded_loader(
        plain_train_set, batch_size=batch_size, shuffle=False, seed=seed
    )
    validation_loader = seeded_loader(
        validation_set, batch_size=batch_size, shuffle=False, seed=seed
    )
    return fit_loader, train_eval_loader, validation_loader


sample_image, sample_label = plain_train_set[0]
print("training examples:", len(plain_train_set))
print("validation examples:", len(validation_set))
print("sample shape:", tuple(sample_image.shape), "label:", sample_label)
print("official CIFAR-10 test split: reserved and not loaded")


## 2. Inspect the provided compact MLP

`CompactMLP` accepts a list of hidden dimensions so width and depth can be
varied without editing the implementation. Dropout has no learnable
parameters. The deterministic diagnostic printed below is the numeric value
submitted on edX; it does not depend on training.


In [ ]:
class CompactMLP(nn.Module):
    def __init__(
        self,
        hidden_dims: tuple[int, ...] = (512, 128),
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        if not hidden_dims or any(width <= 0 for width in hidden_dims):
            raise ValueError("hidden_dims must contain positive widths")
        if not 0.0 <= dropout < 1.0:
            raise ValueError("dropout must be in [0, 1)")

        layers: list[nn.Module] = [nn.Flatten()]
        input_width = 3 * 32 * 32
        for output_width in hidden_dims:
            layers.extend(
                [
                    nn.Linear(input_width, output_width),
                    nn.ReLU(),
                    nn.Dropout(p=dropout),
                ]
            )
            input_width = output_width
        layers.append(nn.Linear(input_width, 10))
        self.network = nn.Sequential(*layers)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        return self.network(images)


def trainable_parameter_count(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


reset_seed()
default_mlp = CompactMLP(hidden_dims=(512, 128), dropout=0.0)
default_parameter_count = trainable_parameter_count(default_mlp)
assert default_parameter_count == 1_640_330
print(default_mlp)
print(f"Deterministic diagnostic — trainable parameters: {default_parameter_count:,}")


**Hand-check the diagnostic before continuing.** The three linear layers have
(3072(512)+512), (512(128)+128), and (128(10)+10) parameters.
ReLU and Dropout contribute zero. The notebook's final Report Values cell will
repeat this total as R2.


## 3. Training, evaluation, and plotting helpers

Every experiment below rebuilds the model, resets the random seed, and
recreates the shuffled loader. This makes comparisons more controlled. It
does **not** make measured metrics suitable for exact grading across every
runtime.


In [ ]:
@dataclass(frozen=True)
class MLPExperiment:
    hidden_dims: tuple[int, ...] = (512, 128)
    dropout: float = 0.0
    weight_decay: float = 0.0
    use_augmentation: bool = False
    learning_rate: float = 1e-3
    batch_size: int = BATCH_SIZE
    epochs: int = EXPERIMENT_EPOCHS
    seed: int = SEED


@torch.no_grad()
def evaluate_classifier(
    model: nn.Module, loader: DataLoader
) -> tuple[float, float]:
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    loss_fn = nn.CrossEntropyLoss(reduction="sum")
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        logits = model(images)
        total_loss += loss_fn(logits, labels).item()
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += labels.numel()
    return total_loss / total_examples, total_correct / total_examples


def run_mlp_experiment(
    label: str, config: MLPExperiment, *, verbose: bool = True
) -> dict:
    reset_seed(config.seed)
    fit_loader, train_eval_loader, validation_loader = make_m1_loaders(
        use_augmentation=config.use_augmentation,
        batch_size=config.batch_size,
        seed=config.seed,
    )
    model = CompactMLP(config.hidden_dims, config.dropout).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
    )
    loss_fn = nn.CrossEntropyLoss()
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "validation_loss": [],
        "validation_accuracy": [],
    }
    started = time.perf_counter()

    for epoch in range(1, config.epochs + 1):
        model.train()
        for images, labels in fit_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(images), labels)
            loss.backward()
            optimizer.step()

        train_loss, train_accuracy = evaluate_classifier(model, train_eval_loader)
        validation_loss, validation_accuracy = evaluate_classifier(
            model, validation_loader
        )
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["validation_loss"].append(validation_loss)
        history["validation_accuracy"].append(validation_accuracy)
        if verbose:
            print(
                f"{label:>18} | epoch {epoch:>2}/{config.epochs} | "
                f"train acc {train_accuracy:6.2%} | "
                f"val acc {validation_accuracy:6.2%}"
            )

    record = {
        "label": label,
        "config": asdict(config),
        "parameter_count": trainable_parameter_count(model),
        "steps_per_epoch": len(fit_loader),
        "seconds": time.perf_counter() - started,
        "history": history,
    }
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return record


def plot_history(record: dict) -> None:
    history = record["history"]
    epochs = np.arange(1, len(history["train_accuracy"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(epochs, history["train_loss"], marker="o", label="train")
    axes[0].plot(
        epochs, history["validation_loss"], marker="o", label="validation"
    )
    axes[0].set(title=f"{record['label']}: loss", xlabel="epoch", ylabel="loss")
    axes[0].legend()

    axes[1].plot(
        epochs, history["train_accuracy"], marker="o", label="train"
    )
    axes[1].plot(
        epochs,
        history["validation_accuracy"],
        marker="o",
        label="validation",
    )
    axes[1].set(
        title=f"{record['label']}: accuracy",
        xlabel="epoch",
        ylabel="accuracy",
        ylim=(0, 1),
    )
    axes[1].legend()
    plt.tight_layout()
    plt.show()


def plot_sweep(records: list[dict], title: str) -> None:
    """Compare sweep trajectories without relying on color alone."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
    styles = [
        ("-", "o"),
        ("--", "s"),
        ("-.", "^"),
        (":", "D"),
    ]

    for record, (line_style, marker) in zip(records, styles):
        history = record["history"]
        epochs = np.arange(1, len(history["train_accuracy"]) + 1)
        train_accuracy = np.asarray(history["train_accuracy"])
        validation_accuracy = np.asarray(
            history["validation_accuracy"]
        )
        common = {
            "label": record["label"],
            "linestyle": line_style,
            "marker": marker,
        }
        axes[0].plot(epochs, train_accuracy, **common)
        axes[1].plot(epochs, validation_accuracy, **common)
        axes[2].plot(
            epochs,
            train_accuracy - validation_accuracy,
            **common,
        )

    axes[0].set(title="Training accuracy", ylabel="accuracy", ylim=(0, 1))
    axes[1].set(title="Validation accuracy", ylabel="accuracy", ylim=(0, 1))
    axes[2].set(
        title="Train - validation gap",
        ylabel="accuracy difference",
    )
    axes[2].axhline(0, color="black", linewidth=0.8)
    for axis in axes:
        axis.set_xlabel("epoch")
        axis.grid(alpha=0.25)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=2,
        frameon=False,
    )
    fig.suptitle(
        f"{title} - single seeded run; measured and ungraded"
    )
    plt.tight_layout(rect=(0, 0.14, 1, 0.93))
    plt.show()


def experiment_summary(records: list[dict]) -> pd.DataFrame:
    rows = []
    for record in records:
        history = record["history"]
        train_accuracy = history["train_accuracy"][-1]
        validation_accuracy = history["validation_accuracy"][-1]
        rows.append(
            {
                "run": record["label"],
                "parameters": record["parameter_count"],
                "steps/epoch": record["steps_per_epoch"],
                "final train acc": train_accuracy,
                "final validation acc": validation_accuracy,
                "train-validation gap": train_accuracy - validation_accuracy,
                "seconds": record["seconds"],
            }
        )
    return pd.DataFrame(rows).set_index("run")


## 4. Establish a baseline

Run the default two-hidden-layer MLP first. Check that loss moves downward,
compare the two accuracy curves, and note whether their gap is widening,
narrowing, or roughly stable. This run is evidence for your reflection, not
an exact graded result.


In [ ]:
baseline_config = MLPExperiment(epochs=BASELINE_EPOCHS)
baseline_result = run_mlp_experiment("baseline", baseline_config)
plot_history(baseline_result)
experiment_summary([baseline_result])


## 5. Sweep capacity

The following runs change only the hidden dimensions. Use the parameter-count
column alongside the curves: a larger hypothesis class may improve training
fit without improving validation performance. Compare the shallow, default,
wide-first-layer, and deeper configurations.


In [ ]:
capacity_configs = {
    "shallow (512)": MLPExperiment(hidden_dims=(512,)),
    "default (512,128)": MLPExperiment(hidden_dims=(512, 128)),
    "wide first (1024,128)": MLPExperiment(hidden_dims=(1024, 128)),
    "deeper (512,256,128)": MLPExperiment(hidden_dims=(512, 256, 128)),
}

capacity_results = [
    run_mlp_experiment(label, config)
    for label, config in capacity_configs.items()
]
plot_sweep(capacity_results, "Capacity comparison")
capacity_table = experiment_summary(capacity_results)
capacity_table


## 6. Test generalization interventions

These runs hold the architecture, optimizer, seed, and training budget fixed.
Each treatment changes one factor relative to the control:

1. dropout probability from 0 to 0.30;
2. AdamW weight decay from 0 to (10^{-4}); or
3. train-only reflected crop plus horizontal flip.

Inspect both validation accuracy and the train−validation gap. A smaller gap
caused only by much worse training fit is not automatically the best model.
The control reuses the identical default run from the capacity sweep instead
of spending time training it again.


### Pause and predict — ungraded

Before running the comparison, which intervention do you expect to produce
the best validation behavior? What do you expect to happen to training
accuracy and the train−validation gap, and why?

You do not submit this response. Pause briefly to form a prediction before
viewing the results.


In [ ]:
generalization_configs = {
    "dropout 0.30": MLPExperiment(dropout=0.30),
    "L2 1e-4": MLPExperiment(weight_decay=1e-4),
    "crop + flip": MLPExperiment(use_augmentation=True),
}

capacity_default = next(
    record
    for record in capacity_results
    if record["label"] == "default (512,128)"
)
generalization_control = {**capacity_default, "label": "control (reused)"}
generalization_results = [
    generalization_control,
    *[
        run_mlp_experiment(label, config)
        for label, config in generalization_configs.items()
    ],
]
plot_sweep(generalization_results, "Generalization interventions")
generalization_table = experiment_summary(generalization_results)
generalization_table


## Pause and reflect — ungraded

Which result most confirmed or changed your prediction? Identify one pattern
in the training and validation curves that supports your interpretation. What
limitation makes that conclusion tentative, and what one-factor experiment
would you try next?

This is a place to pause, not a response field. Your reflection is not graded.


## Further reflection and forum discussion — ungraded

Course staff may return to these questions in the forum. Choose any one to
think about now or discuss later. If you share a response, name the runs you
compared and point to a value or visible curve pattern from your own results.
Do not post code, report values, or answers to the graded decision checks.

1. **Prediction calibration.** What part of your pre-run prediction was
   supported or contradicted? What follow-up would distinguish your explanation
   from ordinary run-to-run variation?
2. **Accuracy versus efficiency.** Find two capacity configurations with
   similar validation behavior but different parameter counts or runtimes.
   Which would you keep under a stated constraint, and what deployment quantity
   did this lab not measure?
3. **Questioning an augmentation assumption.** Crop and horizontal flip assume
   that these image changes preserve the class label. When might that assumption
   fail, and how could you diagnose harm without using the official test split?
4. **Limits of the evidence.** State one conclusion this single seeded split
   supports and one stronger conclusion it does not. What is the smallest
   follow-up that would test stability across seeds, splits, or training budgets?


## Report values

Run the cell below after completing the required experiments. It prints three
fixed, mode-independent integers that verify critical stages of the notebook:
the data split is disjoint, the default model was constructed as specified, and
all distinct required training histories reached their configured epoch count.
Enter R1, R2, and R3 in the first Lab 1 submission problem in MIT Learn.

Measured accuracy, loss, and timing are deliberately absent because they vary
across runtimes and are evidence for interpretation rather than exact answers.

**Submission checklist**

- [ ] I entered R1, R2, and R3 from the report cell.
- [ ] I answered the widening-gap intervention question.
- [ ] I answered the low-and-close-curves capacity question.
- [ ] I did not upload code or submit a run-specific accuracy as an exact value.

**Provenance:** Adapted from `6_7960_Fall_2024_hw1_CIFAR10.ipynb`,
especially its network, training-loop, learning-curve, and augmentation
sections. Implementations are prefilled, the network is deliberately smaller,
and a true validation split replaces tuning on the source notebook's test set.


In [ ]:
trained_results = [
    baseline_result,
    *capacity_results,
    *generalization_results[1:],
]
split_overlap_count = len(set(train_indices) & set(validation_indices))
complete_history_count = sum(
    len(record["history"]["train_accuracy"])
    == record["config"]["epochs"]
    for record in trained_results
)

report_values = {
    "R1 — overlapping train/validation indices": split_overlap_count,
    "R2 — default trainable parameters": default_parameter_count,
    "R3 — complete distinct training histories": complete_history_count,
}
assert tuple(report_values.values()) == (0, 1_640_330, 8)

print("LAB 1 REPORT VALUES (enter integers without separators)")
for label, value in report_values.items():
    print(f"{label}: {value}")
